# 11 — Forward-NaN Bisection v2 (xformers removed, still NaN)

nb10: every init (pure-torch SwiGLU, no xformers) forwards to **NaN** — including old inits with
unrelated weights. nb09: the SAME v5 forwarded **finite (6.66)** in another runtime. So the NaN
is in NeoBERT's forward and is **runtime-dependent**, not a weight/init/xformers problem.

NeoBERT forward ops: embedding → [rotary → QKV → **SDPA attention** → SwiGLU FFN → RMSNorm] ×L
→ final RMSNorm → decoder. SwiGLU is now pure-torch and ruled out. Prime suspects: the CUDA
**SDPA** attention (flash / mem-efficient backend on torch 2.11/cu128) and rotary/RMSNorm.

**Discriminators (verdict in F):**
| Test | If finite | Implicates |
|------|-----------|-----------|
| forward on **CPU** | yes | a CUDA kernel (SDPA backend), not the math |
| forward with **output_attentions=True** (eager attn, no SDPA) | yes | torch SDPA |
| forward under **SDPA MATH backend only** | yes | flash/mem-efficient SDPA kernel |
| per-module hook | — | names the first NaN module |


In [1]:
%%capture
!pip uninstall -y xformers
!pip install -U transformers safetensors huggingface_hub sentencepiece accelerate


In [2]:
import sys, math
from pathlib import Path
import torch
import torch.nn.functional as F
from transformers import AutoModelForMaskedLM, AutoTokenizer

try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e:
    print('Drive mount skipped:', e)
PROJECT_ROOT = Path('/content/drive/MyDrive/SALT3')
sys.path.insert(0, str(PROJECT_ROOT / 'code')); sys.path.insert(0, '/content')
import transformers
print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.version.cuda)

# Which SDPA backends does this torch enable by default?
try:
    from torch.nn.attention import SDPBackend, sdpa_kernel
    HAVE_SDPA_CTX = True
except Exception as e:
    HAVE_SDPA_CTX = False; print('sdpa_kernel ctx unavailable:', e)
print('flash sdp:', torch.backends.cuda.flash_sdp_enabled(),
      '| mem_efficient:', torch.backends.cuda.mem_efficient_sdp_enabled(),
      '| math:', torch.backends.cuda.math_sdp_enabled())

INIT_DIR = PROJECT_ROOT / 'init' / 'videberta_salt_init_v5_globalmap_freqbias' / 'model'

def batch(tok, device):
    s = ['Việt Nam là một quốc gia ở Đông Nam Á.',
         'Hôm nay thời tiết rất đẹp và trời trong xanh.',
         'Kinh tế Việt Nam tăng trưởng trong năm qua.',
         'Trẻ em cần được tiêm phòng đầy đủ để tránh bệnh.']
    return tok(s, padding=True, truncation=True, max_length=32, return_tensors='pt').to(device)

@torch.no_grad()
def fwd_finite(model, enc, **kw):
    o = model(input_ids=enc['input_ids'], attention_mask=enc['attention_mask'], **kw)
    t = o.logits if hasattr(o, 'logits') else o.last_hidden_state
    return bool(torch.isfinite(t).all())


Mounted at /content/drive
torch 2.11.0+cu128 | transformers 5.11.0 | cuda 12.8
flash sdp: True | mem_efficient: True | math: True


## A. Reproduce on CUDA, then test CPU (CUDA-kernel discriminator)

In [3]:
tok = AutoTokenizer.from_pretrained(INIT_DIR, trust_remote_code=True)
m_cuda = AutoModelForMaskedLM.from_pretrained(INIT_DIR, trust_remote_code=True).to('cuda').eval()
ok_cuda = fwd_finite(m_cuda, batch(tok, 'cuda'))
print('CUDA forward finite:', ok_cuda)

m_cpu = AutoModelForMaskedLM.from_pretrained(INIT_DIR, trust_remote_code=True).to('cpu').eval()
ok_cpu = fwd_finite(m_cpu, batch(tok, 'cpu'))
print('CPU  forward finite:', ok_cpu,
      '  <-- if True while CUDA False, it is a CUDA SDPA-backend kernel bug' if (ok_cpu and not ok_cuda) else '')


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

CUDA forward finite: False


Loading weights:   0%|          | 0/172 [00:00<?, ?it/s]

CPU  forward finite: True   <-- if True while CUDA False, it is a CUDA SDPA-backend kernel bug


## B. Eager attention (output_attentions=True) vs SDPA on CUDA

In [4]:
# output_attentions=True routes NeoBERT through its manual eager attention (no SDPA call).
try:
    ok_eager = fwd_finite(m_cuda, batch(tok, 'cuda'), output_attentions=True)
except Exception as e:
    ok_eager = f'err: {type(e).__name__}: {e}'
print('CUDA eager-attention forward finite:', ok_eager,
      '  <-- if True, torch SDPA is the culprit' if ok_eager is True and not ok_cuda else '')


CUDA eager-attention forward finite: False 


## C. Force SDPA MATH backend (isolate flash / mem-efficient kernels)

In [5]:
results = {}
if HAVE_SDPA_CTX:
    for name, backend in [('MATH', SDPBackend.MATH),
                          ('FLASH', SDPBackend.FLASH_ATTENTION),
                          ('MEM_EFFICIENT', SDPBackend.EFFICIENT_ATTENTION)]:
        try:
            with sdpa_kernel(backend):
                results[name] = fwd_finite(m_cuda, batch(tok, 'cuda'))
        except Exception as e:
            results[name] = f'err: {type(e).__name__}'
    for k, v in results.items():
        print(f'  SDPA backend {k:14s} forward finite: {v}')
else:
    # legacy global toggles
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(False)
    torch.backends.cuda.enable_math_sdp(True)
    results['MATH(global)'] = fwd_finite(m_cuda, batch(tok, 'cuda'))
    print('  MATH-only (global toggles) forward finite:', results['MATH(global)'])


  SDPA backend MATH           forward finite: False
  SDPA backend FLASH          forward finite: err: RuntimeError
  SDPA backend MEM_EFFICIENT  forward finite: False


/root/.cache/huggingface/modules/transformers_modules/model/4d2fce8e30ee53fb/model.py:199: UserWarning: Memory efficient kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:986.)
  attn = scaled_dot_product_attention(
/root/.cache/huggingface/modules/transformers_modules/model/4d2fce8e30ee53fb/model.py:199: UserWarning: Memory Efficient attention has been runtime disabled. (Triggered internally at /pytorch/aten/src/ATen/native/transformers/sdp_utils_cpp.h:552.)
  attn = scaled_dot_product_attention(
/root/.cache/huggingface/modules/transformers_modules/model/4d2fce8e30ee53fb/model.py:199: UserWarning: Flash attention kernel not used because: (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/sdp_utils.cpp:988.)
  attn = scaled_dot_product_attention(
/root/.cache/huggingface/modules/transformers_modules/model/4d2fce8e30ee53fb/model.py:199: UserWarning: Flash Attention does not support non-null attn_mask. (

## D. Per-module first-NaN hook (CUDA)

In [6]:
first = []
def mk(n):
    def h(mod, inp, out):
        t = out[0] if isinstance(out, tuple) else out
        if torch.is_tensor(t) and not torch.isfinite(t).all():
            first.append((n, type(mod).__name__))
    return h
hs = [m.register_forward_hook(mk(n)) for n, m in m_cuda.named_modules() if n]
_ = fwd_finite(m_cuda, batch(tok, 'cuda'))
for h in hs: h.remove()
print('first 10 NaN-emitting modules (execution order):')
for n, t in first[:10]: print(f'   {n}  ({t})')
print('-> NaN born in:', first[0] if first else '(none)')


first 10 NaN-emitting modules (execution order):
   model.transformer_encoder.0.wo  (Linear)
   model.transformer_encoder.0.ffn_norm  (RMSNorm)
   model.transformer_encoder.0.ffn.w12  (Linear)
   model.transformer_encoder.0.ffn.w3  (Linear)
   model.transformer_encoder.0.ffn  (SwiGLU)
   model.transformer_encoder.0  (EncoderBlock)
   model.transformer_encoder.1.attention_norm  (RMSNorm)
   model.transformer_encoder.1.qkv  (Linear)
   model.transformer_encoder.1.wo  (Linear)
   model.transformer_encoder.1.ffn_norm  (RMSNorm)
-> NaN born in: ('model.transformer_encoder.0.wo', 'Linear')


## E. Inspect the attention mask + a raw SDPA call with NeoBERT's exact mask shape

In [7]:
enc = batch(tok, 'cuda'); am = enc['attention_mask']
print('attention_mask per-row sums (real tokens):', am.sum(1).tolist())
H = m_cuda.config.num_attention_heads
# NeoBERT expands: [B,L] -> [B,H,L,L] then .bool()
exp = am.unsqueeze(1).unsqueeze(1).repeat(1, H, am.size(-1), 1).bool()
allfalse_rows = (~exp).all(-1).sum().item()
print('fully-masked query rows in expanded mask (all-False -> SDPA NaN):', allfalse_rows)
B, L = am.shape; d = m_cuda.config.hidden_size // H
q = torch.randn(B, H, L, d, device='cuda'); k = torch.randn_like(q); v = torch.randn_like(q)
out = F.scaled_dot_product_attention(q, k, v, attn_mask=exp)
print('raw SDPA(random q,k,v, NeoBERT mask) finite:', bool(torch.isfinite(out).all()))


attention_mask per-row sums (real tokens): [14, 13, 12, 14]
fully-masked query rows in expanded mask (all-False -> SDPA NaN): 0
raw SDPA(random q,k,v, NeoBERT mask) finite: True


## F. Verdict + fix

In [8]:
print('=' * 70); print('FORWARD-NAN BISECTION v2 — VERDICT'); print('=' * 70)
print(f'torch {torch.__version__} cuda {torch.version.cuda}')
print(f'CUDA finite {ok_cuda} | CPU finite {ok_cpu} | eager finite {ok_eager}')
print(f'SDPA backends: {results}')
print(f'first NaN module: {first[0] if first else "(none)"}')
print('-' * 70)
math_ok = results.get('MATH', results.get('MATH(global)')) is True
if not ok_cuda and ok_cpu:
    print('CUDA-only NaN -> a CUDA attention kernel. ', end='')
if not ok_cuda and (ok_eager is True or math_ok):
    print('FIX: force the SDPA MATH backend (or eager attn) in NeoBERT forward.')
    print('Implement: wrap encoder forward in `with sdpa_kernel(SDPBackend.MATH):` via a')
    print('model.py patch, OR globally disable flash/mem-efficient sdp at load time.')
elif not ok_cuda and not ok_cpu:
    print('NaN on CPU too -> not a CUDA kernel; suspect the masked-softmax / rotary / RMSNorm')
    print('at the first-NaN module above. Inspect that op.')
else:
    print('forward finite here -> NaN was transient runtime state; pin torch+disable flash sdp.')
print('=' * 70)


FORWARD-NAN BISECTION v2 — VERDICT
torch 2.11.0+cu128 cuda 12.8
CUDA finite False | CPU finite True | eager finite False
SDPA backends: {'MATH': False, 'FLASH': 'err: RuntimeError', 'MEM_EFFICIENT': False}
first NaN module: ('model.transformer_encoder.0.wo', 'Linear')
----------------------------------------------------------------------
CUDA-only NaN -> a CUDA attention kernel. forward finite here -> NaN was transient runtime state; pin torch+disable flash sdp.
